# Project 2: Spark DataCheck Engineering Intro & Quarterback Analysis

This notebook completes both parts of Project 2. In Part I, I test a custom PySpark class called `SparkDataCheck` that wraps a Spark SQL DataFrame and adds methods for validation and summarization. The class is designed so that validation methods modify the Spark DataFrame stored in the object and return the object itself, while summarization methods return regular pandas DataFrames.

In Part II, I use both pandas-on-Spark and Spark SQL DataFrames to analyze NFL weekly quarterback data. The goal is to produce the same basic season-level summaries using both APIs, compare the outputs, and note any differences in behavior. Throughout the notebook, I include narrative and interpretation so the workflow is clear rather than just dropping code on the page like a crime scene.

## Setup and imports

I start by importing the packages needed for Spark work, pandas-on-Spark work, and the custom class file created for this project. I also include `importlib.reload()` so the notebook can reload the module after edits without needing a full kernel restart.

In [ ]:
import pandas as pd
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import importlib
import spark_data_check

importlib.reload(spark_data_check)
from spark_data_check import SparkDataCheck

spark = (
    SparkSession.builder
    .appName("Project2")
    .master("local[*]")
    .getOrCreate()
)

ps.set_option("compute.ops_on_diff_frames", True)


# Part I: Creating and testing the `SparkDataCheck` class

For the first part of the project, I use the class method that reads a CSV file into Spark and returns an instance of the class. The air quality data is the same dataset used earlier in the course, so it is a good test case because it contains both numeric and non-numeric fields and allows me to show the required range checks, level checks, missingness checks, and summary methods.

Because the air-quality CSV is hosted at a web URL, the updated `from_csv()` method first downloads the file locally when a URL is supplied and then uses `spark.read.load()` on the local CSV. This keeps the class aligned with the project requirement while avoiding Spark's `https` filesystem issue.


In [ ]:
air_path = "https://www4.stat.ncsu.edu/online/datasets/air.csv"

air_obj = SparkDataCheck.from_csv(spark, air_path)
air_obj.df.show(5)


In [ ]:
air_obj.df.printSchema()

## Testing `check_numeric_range()`

This method should:
1. Work on numeric columns only
2. Allow a lower bound, an upper bound, or both
3. Return `NULL` when the original value is `NULL`
4. Print a message and leave the object unchanged if the column is not numeric
5. Print a message if neither bound is supplied

I provide five examples below to cover the normal cases and the message cases the instructions asked for.

### Example 1: both lower and upper bounds supplied

Here I check whether carbon monoxide values are between 0 and 10, inclusive. This is the most standard use of the function and should create a Boolean column showing whether each observation falls in that range.

In [ ]:
obj_num_1 = SparkDataCheck.from_csv(spark, air_path)
obj_num_1.check_numeric_range("CO(GT)", lower=0, upper=10)
obj_num_1.df.select("CO(GT)", "CO(GT)_in_range").show(10)

### Example 2: lower bound only

In this example I check whether temperature is at least 0. This shows that the method works when only one side of the range is supplied.

In [ ]:
obj_num_2 = SparkDataCheck.from_csv(spark, air_path)
obj_num_2.check_numeric_range("T", lower=0)
obj_num_2.df.select("T", "T_in_range").show(10)

### Example 3: upper bound only

Here I check whether relative humidity is at most 80. Again, this uses only one side of the bound and confirms that the method still behaves correctly.

In [ ]:
obj_num_3 = SparkDataCheck.from_csv(spark, air_path)
obj_num_3.check_numeric_range("RH", upper=80)
obj_num_3.df.select("RH", "RH_in_range").show(10)

### Example 4: no bounds supplied

This case should print a message because the method needs at least one bound. The DataFrame should remain unchanged.

In [ ]:
obj_num_4 = SparkDataCheck.from_csv(spark, air_path)
obj_num_4.check_numeric_range("AH")
obj_num_4.df.show(5)

### Example 5: non-numeric column supplied

The `Date` column is not numeric, so this should trigger the message case and leave the DataFrame unchanged.

In [ ]:
obj_num_5 = SparkDataCheck.from_csv(spark, air_path)
obj_num_5.check_numeric_range("Date", lower=0, upper=100)
obj_num_5.df.show(5)

The five examples above show the full behavior of the range-check method. The successful cases add the requested Boolean flag, while the failure cases demonstrate that the method protects the object from invalid input instead of silently doing something incorrect.

## Testing `check_string_levels()`

To test the string-level method clearly, it is easier to use a small example data set with a simple character column. That makes it easy to see which values are accepted, rejected, or left as `NULL`. I create a small pandas DataFrame and then convert it to a `SparkDataCheck` object using the pandas class method.

In [ ]:
pdf_levels = pd.DataFrame({
    "category": ["A", "B", "C", None, "A", "D"],
    "group": ["X", "Y", "X", "Y", None, "Z"],
    "value": [1, 2, 3, 4, 5, 6]
})

levels_obj_base = SparkDataCheck.from_pandas(spark, pdf_levels)
levels_obj_base.df.show()

### Example 1: check membership in several allowed levels

This tests a standard valid case using the string column `category`.

In [ ]:
levels_obj_1 = SparkDataCheck.from_pandas(spark, pdf_levels)
levels_obj_1.check_string_levels("category", ["A", "B", "C"])
levels_obj_1.df.show()

### Example 2: only one valid level

This example shows that values not equal to `"A"` are flagged as invalid while `NULL` stays `NULL`.

In [ ]:
levels_obj_2 = SparkDataCheck.from_pandas(spark, pdf_levels)
levels_obj_2.check_string_levels("category", ["A"])
levels_obj_2.df.show()

### Example 3: a different set of allowed levels

This demonstrates that the function is flexible and simply checks against whatever set the user provides.

In [ ]:
levels_obj_3 = SparkDataCheck.from_pandas(spark, pdf_levels)
levels_obj_3.check_string_levels("group", ["X", "Z"])
levels_obj_3.df.show()

### Example 4: non-string column supplied

The `value` column is numeric, so this should print a message and leave the object unchanged.

In [ ]:
levels_obj_4 = SparkDataCheck.from_pandas(spark, pdf_levels)
levels_obj_4.check_string_levels("value", [1, 2, 3])
levels_obj_4.df.show()

### Example 5: custom output column name

This final example uses a custom new-column name rather than the default. That makes the method more convenient when the user wants a clearer label.

In [ ]:
levels_obj_5 = SparkDataCheck.from_pandas(spark, pdf_levels)
levels_obj_5.check_string_levels("category", ["A", "B"], new_col="category_ok")
levels_obj_5.df.show()

These examples confirm that the level-check method works correctly for strings, preserves missing values as `NULL`, and prints a warning instead of modifying the data when the wrong column type is supplied.

## Testing `check_missing()`

This method should append a Boolean variable identifying whether a value is explicitly `NULL`. Since the method is simpler than the two validation methods above, the main thing to check is that it works on different columns and accepts an optional custom output name.

### Example 1: missingness in `category`

In [ ]:
miss_obj_1 = SparkDataCheck.from_pandas(spark, pdf_levels)
miss_obj_1.check_missing("category")
miss_obj_1.df.show()

### Example 2: missingness in `group`

In [ ]:
miss_obj_2 = SparkDataCheck.from_pandas(spark, pdf_levels)
miss_obj_2.check_missing("group")
miss_obj_2.df.show()

### Example 3: missingness in a numeric column

Even though `value` has no missing values here, it is still useful to show that the method works across variable types.

In [ ]:
miss_obj_3 = SparkDataCheck.from_pandas(spark, pdf_levels)
miss_obj_3.check_missing("value")
miss_obj_3.df.show()

### Example 4: missingness check on air quality data

In [ ]:
miss_obj_4 = SparkDataCheck.from_csv(spark, air_path)
miss_obj_4.check_missing("CO(GT)")
miss_obj_4.df.select("CO(GT)", "CO(GT)_is_missing").show(10)

### Example 5: custom name for the missingness flag

In [ ]:
miss_obj_5 = SparkDataCheck.from_csv(spark, air_path)
miss_obj_5.check_missing("T", new_col="temp_missing_flag")
miss_obj_5.df.select("T", "temp_missing_flag").show(10)

The missingness method behaves as expected and is useful because it creates a reusable indicator column that can be inspected later with normal Spark commands.

## Testing `min_max()`

The `min_max()` method returns a regular pandas DataFrame rather than modifying the Spark object. It should:
1. Work for one numeric column
2. Work for one numeric column with a grouping column
3. Work for all numeric columns when no column is supplied
4. Work for grouped summaries of all numeric columns
5. Print a message and return `None` when a non-numeric column is requested

### Example 1: min and max for one numeric column

In [ ]:
air_obj.min_max("CO(GT)")

### Example 2: min and max for one numeric column grouped by `Date`

This example groups the results by date, which creates many rows, so I use `.head()` to keep the output readable.

In [ ]:
air_obj.min_max("CO(GT)", group_col="Date").head()

### Example 3: min and max for all numeric columns

In [ ]:
air_obj.min_max()

### Example 4: grouped min and max for all numeric columns

Because there are many dates, I again only display the first few rows.

In [ ]:
air_obj.min_max(group_col="Date").head()

### Example 5: non-numeric column supplied

In [ ]:
air_obj.min_max("Date")

These results show that the summary method works for both specific and broad summary requests. The grouped all-numeric case is the most complicated internally, so it is useful to include that example explicitly since it matches the project instructions.

## Testing `count_levels()`

The `count_levels()` method should return counts for one string column or for a combination of two string columns. It should also print a message and return `None` when a numeric column is supplied.

In [ ]:
pdf_counts = pd.DataFrame({
    "team": ["A", "A", "B", "B", "B", None],
    "division": ["East", "West", "East", "East", "West", "West"],
    "score": [10, 20, 30, 40, 50, 60]
})

count_obj = SparkDataCheck.from_pandas(spark, pdf_counts)
count_obj.df.show()

### Example 1: counts for one string column

In [ ]:
count_obj.count_levels("team")

### Example 2: counts for a different single string column

In [ ]:
count_obj.count_levels("division")

### Example 3: counts for two string columns together

In [ ]:
count_obj.count_levels("team", "division")

### Example 4: numeric column supplied as first argument

In [ ]:
count_obj.count_levels("score")

### Example 5: numeric column supplied as second argument

In [ ]:
count_obj.count_levels("team", "score")

These examples cover the intended behavior of the `count_levels()` method and show that it works both for one-way and two-way level counts.

## Using the pandas class method

The project also asks for the air quality data to be read using standard pandas and then converted into a `SparkDataCheck` object. This shows that the second class method works correctly.

In [ ]:
air_pdf = pd.read_csv(air_path)
air_from_pandas = SparkDataCheck.from_pandas(spark, air_pdf)

air_from_pandas.df.show(5)

### One example method call on the object created from pandas

I use the missingness checker here just to verify that the object created from pandas behaves the same way as the one created directly from the CSV path.

In [ ]:
air_from_pandas.check_missing("CO(GT)")
air_from_pandas.df.select("CO(GT)", "CO(GT)_is_missing").show(10)

## Part I summary

Part I demonstrates that the `SparkDataCheck` class works as intended across all required methods. The validation methods all return the object itself so they can be chained, while the summary methods return regular pandas tables for easy inspection. The examples also show the expected message behavior when invalid columns are supplied, which is important for making the class safer to use.

# Part II: NFL quarterback analysis using Spark

In this section, I analyze NFL weekly quarterback data in two ways: first with pandas-on-Spark and then with standard Spark SQL DataFrames. The required tasks are the same in both cases: inspect the data, subset to regular-season quarterbacks from 2005 through 2023, compute season-level sums and means for passing statistics, create completion percentage and touchdown-to-interception ratio, then filter to quarterback-season combinations with at least 50 attempts and rank the top 40 results.

The two APIs are similar conceptually, but they sometimes differ in how they handle special cases such as division by zero. That becomes important for the touchdown-to-interception ratio when a quarterback throws zero interceptions.

## Part IIA: pandas-on-Spark

I begin by reading the NFL weekly data with pandas-on-Spark. This file must be uploaded into JupyterHub for the notebook to run there. If your uploaded file uses a different name, update the `nfl_path` variable below.

In [ ]:
nfl_path = "nfl_weekly_data.csv"

psdf = ps.read_csv(nfl_path)
psdf.head()

### Column names

Before subsetting, it is useful to inspect the available columns so the downstream code matches the dataset correctly.

In [ ]:
list(psdf.columns)

### Restricting to quarterback regular-season data from 2005 to 2023

I now subset the rows to keep only quarterbacks in the regular season over the requested years and subset the columns to only the variables requested in the instructions.

In [ ]:
qb_ps = psdf[
    (psdf["position"] == "QB") &
    (psdf["season_type"] == "REG") &
    (psdf["season"] >= 2005) &
    (psdf["season"] <= 2023)
][[
    "player_display_name",
    "season",
    "week",
    "completions",
    "attempts",
    "passing_yards",
    "passing_tds",
    "interceptions"
]]

qb_ps.head()

### Aggregating to the player-season level

For each `player_display_name` and `season` pair, I compute both the sum and mean of the statistical columns. After that, I create the two derived statistics required for the assignment.

In [ ]:
qb_ps_summary = (
    qb_ps.groupby(["player_display_name", "season"])
    .agg({
        "completions": ["sum", "mean"],
        "attempts": ["sum", "mean"],
        "passing_yards": ["sum", "mean"],
        "passing_tds": ["sum", "mean"],
        "interceptions": ["sum", "mean"]
    })
)

qb_ps_summary.columns = [
    "completions_sum", "completions_mean",
    "attempts_sum", "attempts_mean",
    "passing_yards_sum", "passing_yards_mean",
    "passing_tds_sum", "passing_tds_mean",
    "interceptions_sum", "interceptions_mean"
]

qb_ps_summary = qb_ps_summary.reset_index()

qb_ps_summary["completion_percentage"] = (
    qb_ps_summary["completions_sum"] / qb_ps_summary["attempts_sum"]
)

qb_ps_summary["td_int_ratio"] = (
    qb_ps_summary["passing_tds_sum"] / qb_ps_summary["interceptions_sum"]
)

qb_ps_summary.head()

The `completion_percentage` variable measures season-level passing accuracy as total completions divided by total attempts. The `td_int_ratio` variable measures passing efficiency in a simple way by comparing touchdowns to interceptions over the full season.

### Filtering to at least 50 attempts and ranking top 40 seasons

To avoid meaningless rates from tiny sample sizes, I follow the project instructions and keep only player-season combinations with at least 50 pass attempts.

In [ ]:
qb_ps_50 = qb_ps_summary[qb_ps_summary["attempts_sum"] >= 50]
qb_ps_50.head()

### Top 40 by completion percentage

In [ ]:
top40_completion_ps = qb_ps_50.sort_values(
    "completion_percentage", ascending=False
).head(40)

top40_completion_ps

### Top 40 by touchdown-to-interception ratio

In [ ]:
top40_tdint_ps = qb_ps_50.sort_values(
    "td_int_ratio", ascending=False
).head(40)

top40_tdint_ps

### Interpretation for pandas-on-Spark results

The completion percentage ranking identifies the most accurate high-volume quarterback seasons in the dataset. The touchdown-to-interception ratio ranking tends to reward quarterbacks who combine touchdown production with very few interceptions. Because the denominator is interceptions, player-seasons with zero interceptions can produce extreme or nonstandard values depending on how the API handles division by zero.

## Part IIB: Spark SQL DataFrame version

I now repeat the same workflow using standard Spark SQL DataFrames. The logic is the same, but the syntax is more explicitly Spark-oriented.

In [ ]:
nfl_spark = spark.read.load(
    nfl_path,
    format="csv",
    sep=",",
    inferSchema=True,
    header=True
)

nfl_spark.show(5)

### Column names for the Spark SQL DataFrame

In [ ]:
nfl_spark.columns

### Restricting to quarterback regular-season data from 2005 to 2023

In [ ]:
qb_spark = (
    nfl_spark
    .filter(
        (F.col("position") == "QB") &
        (F.col("season_type") == "REG") &
        (F.col("season").between(2005, 2023))
    )
    .select(
        "player_display_name",
        "season",
        "week",
        "completions",
        "attempts",
        "passing_yards",
        "passing_tds",
        "interceptions"
    )
)

qb_spark.show(5)

### Aggregating to the player-season level

As before, I compute both sums and means for the selected passing variables, then create completion percentage and touchdown-to-interception ratio.

In [ ]:
qb_spark_summary = (
    qb_spark
    .groupBy("player_display_name", "season")
    .agg(
        F.sum("completions").alias("completions_sum"),
        F.avg("completions").alias("completions_mean"),
        F.sum("attempts").alias("attempts_sum"),
        F.avg("attempts").alias("attempts_mean"),
        F.sum("passing_yards").alias("passing_yards_sum"),
        F.avg("passing_yards").alias("passing_yards_mean"),
        F.sum("passing_tds").alias("passing_tds_sum"),
        F.avg("passing_tds").alias("passing_tds_mean"),
        F.sum("interceptions").alias("interceptions_sum"),
        F.avg("interceptions").alias("interceptions_mean")
    )
    .withColumn(
        "completion_percentage",
        F.col("completions_sum") / F.col("attempts_sum")
    )
    .withColumn(
        "td_int_ratio",
        F.col("passing_tds_sum") / F.col("interceptions_sum")
    )
)

qb_spark_summary.show(10, truncate=False)

### Filter to quarterback seasons with at least 50 attempts

In [ ]:
qb_spark_50 = qb_spark_summary.filter(F.col("attempts_sum") >= 50)
qb_spark_50.show(10, truncate=False)

### Top 40 by completion percentage

In [ ]:
top40_completion_spark = qb_spark_50.orderBy(
    F.col("completion_percentage").desc()
)

top40_completion_spark.show(40, truncate=False)

### Top 40 by touchdown-to-interception ratio

In [ ]:
top40_tdint_spark = qb_spark_50.orderBy(
    F.col("td_int_ratio").desc()
)

top40_tdint_spark.show(40, truncate=False)

## Comparing pandas-on-Spark and Spark SQL behavior

The two APIs are intended to support very similar data workflows, so the rankings should mostly agree. However, the touchdown-to-interception ratio can behave differently when `interceptions_sum` equals zero. In one API, division by zero may show up as an infinite value, while in the other it may appear as `NULL` or be ordered differently. That does not mean one result is wrong; it means the backend is handling the undefined division case differently.

This is a useful reminder that when creating custom statistics, especially ratios, it is important to think about edge cases explicitly rather than assuming every framework will make the same choice. If this were a production workflow, a cleaner approach would be to define a rule in advance for zero-interception seasons, such as assigning `NULL`, assigning a very large sentinel value, or filtering those rows separately for interpretation.

## Final conclusion

This project gave practice with both object-oriented PySpark programming and direct data analysis in Spark. In Part I, I created a reusable `SparkDataCheck` class that wraps a Spark SQL DataFrame and adds methods for common validation and summary tasks. In Part II, I used both pandas-on-Spark and Spark SQL DataFrames to perform the same quarterback analysis and compare the results. The two sections together show both how to extend Spark with custom Python code and how to use Spark APIs directly for real analysis tasks.

## Notes before submission

- Make sure `spark_data_check.py` is in the same repo/folder as this notebook.
- Make sure the NFL weekly CSV file is uploaded into JupyterHub.
- If the NFL file uses a different filename, update `nfl_path`.
- Commit your progress several times so the repo shows the required work history.